# 406 MHz Alert Triage: Colab Training Notebook

Fine-tunes a multilingual transformer (default `xlm-roberta-base`) for the **advisory** alert-priority classifier,
selects the operating threshold under a recall gate, and summarizes the results.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. Have `alert_triage.zip` ready (upload it when prompted, or put it in Google Drive).
3. **Data privacy:** use *synthetic* data here. Do **not** upload real RCC/MCC alert text unless your data-sharing
   agreement explicitly allows Google Colab. For real data, run the same code in an approved environment.

Synthetic results only prove the pipeline works. They are not evidence of real-world performance or safety.

## 1. Configuration

In [1]:
#@title Settings { run: "auto" }
USE_DRIVE = True            #@param {type:"boolean"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/alert_triage/alert_triage.zip"  #@param {type:"string"}
DRIVE_OUT_DIR  = "/content/drive/MyDrive/alert_triage/results"           #@param {type:"string"}
REUPLOAD_ZIP = True          #@param {type:"boolean"}  # delete any old zip so you always run the latest code

DATA_SOURCE = "synthetic"    #@param ["synthetic", "upload"]
N_SYNTH = 20000              #@param {type:"integer"}

MODEL_NAME = "xlm-roberta-base"  #@param ["xlm-roberta-base", "microsoft/mdeberta-v3-base", "bert-base-multilingual-cased"]
MAX_LEN = 256                #@param {type:"integer"}
EPOCHS = 3                   #@param {type:"integer"}
BATCH_SIZE = 16              #@param {type:"integer"}
POS_BOOST = 2.0              #@param {type:"number"}
TARGET_RECALL = 0.99         #@param {type:"number"}
ALPHA = 0.05                 #@param {type:"number"}
RUN_BASELINE = True          #@param {type:"boolean"}

## 2. Check the GPU and define a helper

In [2]:
import subprocess, sys, torch

def sh(cmd):
    """Run a shell command, stream its output into the cell, raise on failure."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Use Runtime > Change runtime type > T4 GPU (training on CPU would be very slow).")

CUDA available: True
GPU: Tesla T4


## 3. Get the project (upload, or from Google Drive)

In [3]:
import glob, os, shutil

os.chdir("/content")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.exists(DRIVE_ZIP_PATH), f"Not found: {DRIVE_ZIP_PATH}"
    shutil.copy(DRIVE_ZIP_PATH, "/content/alert_triage.zip")
else:
    if REUPLOAD_ZIP:
        for old_zip in glob.glob("/content/alert_triage*.zip"):
            os.remove(old_zip)
    if not glob.glob("/content/alert_triage*.zip"):
        from google.colab import files
        print("Select alert_triage.zip ...")
        files.upload()          # saved into /content

zips = sorted(glob.glob("/content/alert_triage*.zip"))
assert zips, "alert_triage.zip not found. Upload it or enable USE_DRIVE."
sh(f'unzip -q -o "{zips[-1]}" -d /content')
os.chdir("/content/alert_triage")
# Guard against a stale zip from an earlier session (older versions had a transformers 5.x incompatibility)
assert "build_training_kwargs" in open("triage/train_bert.py").read(),     "This alert_triage.zip is out of date. Upload the latest one (set REUPLOAD_ZIP = True)."
print("Project ready in", os.getcwd())
sh("ls")

Mounted at /content/drive
Project ready in /content/alert_triage
Dockerfile
docs
Makefile
pytest.ini
README.md
requirements-dev.txt
requirements-train.txt
requirements.txt
tests
triage


## 4. Install dependencies (Colab already has torch and transformers)

In [4]:
sh("pip install -q datasets accelerate scikit-learn scipy pandas joblib fastapi 'pydantic>=2' pytest httpx")
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

transformers 5.18.0 | datasets 4.8.5


## 5. Run the unit tests (quick sanity check)

In [5]:
sh("python -m pytest -q")

...................                                                      [100%]
19 passed in 27.55s


## 6. Data
`synthetic` generates test data. `upload` expects three CSVs (`train.csv`, `val.csv`, `test.csv`) in the format described in
`docs/DATA_REQUIREMENTS.md`. Read the privacy note at the top before using real data.

In [6]:
os.makedirs("data", exist_ok=True)
if DATA_SOURCE == "synthetic":
    sh(f"python -m triage.synth --n {N_SYNTH} --out data")
else:
    from google.colab import files
    print("Upload train.csv, val.csv and test.csv ...")
    up = files.upload()
    for name in up:
        shutil.move(name, os.path.join("data", os.path.basename(name)))
    missing = [f for f in ("train.csv", "val.csv", "test.csv") if not os.path.exists(f"data/{f}")]
    assert not missing, f"Missing files: {missing}"

import pandas as pd
for s in ("train", "val", "test"):
    d = pd.read_csv(f"data/{s}.csv")
    print(f"{s:5s} n={len(d):6d}  positives={int(d.label.sum()):5d} ({d.label.mean():.1%})")

train: n=14001 positives=1025 (7.3%)
val: n=3001 positives=525 (17.5%)
test: n=2998 positives=651 (21.7%)
SYNTHETIC DATA: for pipeline testing only. Results are not evidence of safety.
train n= 14001  positives= 1025 (7.3%)
val   n=  3001  positives=  525 (17.5%)
test  n=  2998  positives=  651 (21.7%)


## 7. Baseline (TF-IDF + logistic regression, same recall gate)

In [7]:
if RUN_BASELINE:
    sh(f"python -m triage.baseline --data data --out artifacts/baseline --target-recall {TARGET_RECALL} --alpha {ALPHA}")
else:
    print("Skipped.")

{
  "model_kind": "baseline",
  "model_path": "artifacts/baseline/model.joblib",
  "model_version": "c3effa35e3cd",
  "threshold": 0.1017312729344579,
  "gate_met_on_val": true,
  "target_recall": 0.99,
  "alpha": 0.05,
  "min_positives_required": 299,
  "val": {
    "threshold": 0.1017312729344579,
    "n_pos": 525,
    "n_neg": 2476,
    "recall": 0.9980952380952381,
    "recall_lower_bound": 0.9909962220915837,
    "missed_distress": 1,
    "low_priority_fraction_of_false": 0.8865105008077544
  },
  "test": {
    "threshold": 0.1017312729344579,
    "n_pos": 651,
    "n_neg": 2347,
    "recall": 0.9969278033794163,
    "recall_lower_bound": 0.9903609008109158,
    "missed_distress": 2,
    "low_priority_fraction_of_false": 0.8922028121005539
  },
  "test_auroc": 0.9922799115385396,
  "negation_pair_pass_rate": 1.0,
  "slices_failing": 6,
  "slices_insufficient": 7
}


## 8. Fine-tune the transformer
Class-weighted loss, temperature scaling on validation, and a threshold chosen on validation under the recall gate.
If you hit a CUDA out-of-memory error, lower `BATCH_SIZE` (for example to 8) or `MAX_LEN`.

In [8]:
sh(f"python -m triage.train_bert --data data --out artifacts/bert --model {MODEL_NAME} "
   f"--max-len {MAX_LEN} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --pos-boost {POS_BOOST} "
   f"--target-recall {TARGET_RECALL} --alpha {ALPHA}")


Map: 100%|██████████| 14001/14001 [00:01<00:00, 7512.02 examples/s]

Map: 100%|██████████| 3001/3001 [00:00<00:00, 7603.29 examples/s]

Map: 100%|██████████| 2998/2998 [00:00<00:00, 7538.16 examples/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7465.94it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

## 9. Results summary

In [9]:
import json, os
import pandas as pd
from IPython.display import display, Markdown

def load_op(kind):
    p = f"artifacts/{kind}/operating_point.json"
    return json.load(open(p)) if os.path.exists(p) else None

rows = []
for kind in ("baseline", "bert"):
    op = load_op(kind)
    if op is None:
        continue
    t = op["test"]
    cleared = bool(op["gate_met_on_val"] and t["recall_lower_bound"] >= op["target_recall"] and op["slices_failing"] == 0)
    rows.append({
        "model": kind,
        "gate met (val)": op["gate_met_on_val"],
        "threshold": round(op["threshold"], 4),
        "test recall": round(t["recall"], 4),
        "recall lower bound": round(t["recall_lower_bound"], 4),
        "missed distress": t["missed_distress"],
        "false alerts low-priority": round(t["low_priority_fraction_of_false"], 3),
        "negation pass": op.get("negation_pair_pass_rate"),
        "slices FAIL": op["slices_failing"],
        "slices INSUFFICIENT": op["slices_insufficient"],
        "meets release criteria": cleared,
    })

if not rows:
    display(Markdown("**No results found.** Run the training cells first."))
else:
    summary = pd.DataFrame(rows).set_index("model")
    display(Markdown("### Model comparison on the held-out test set"))
    display(summary.T)

    for kind in ("baseline", "bert"):
        f = f"artifacts/{kind}/slice_report_test.csv"
        if os.path.exists(f):
            sl = pd.read_csv(f)
            colour = lambda v: {"FAIL": "background-color:#f8c9c9", "PASS": "background-color:#c9f0d2",
                                "INSUFFICIENT_DATA": "background-color:#fff2b3"}.get(v, "")
            styler = sl.style.map(colour, subset=["gate"]) if hasattr(sl.style, "map") else sl.style.applymap(colour, subset=["gate"])
            display(Markdown(f"### Slice gate: {kind}"))
            display(styler.format({"recall": "{:.3f}", "recall_lb": "{:.3f}", "low_priority_frac_of_false": "{:.3f}"}))

    notes = []
    if globals().get("DATA_SOURCE", "synthetic") == "synthetic":
        notes.append("**These results come from SYNTHETIC data.** They only confirm that the pipeline runs end to end. They say nothing about real-world performance or safety.")
    for r in rows:
        if not r["gate met (val)"]:
            notes.append(f"**{r['model']}:** recall gate NOT met on validation. Threshold fell back to 0.0 (flag everything). Not deployable.")
        if r["slices FAIL"]:
            notes.append(f"**{r['model']}:** {r['slices FAIL']} slice(s) FAIL the recall gate. Each is a release blocker.")
        if r["slices INSUFFICIENT"]:
            notes.append(f"**{r['model']}:** {r['slices INSUFFICIENT']} slice(s) have too few positives to prove the gate. Collect more data or document the risk.")
        if r["negation pass"] is not None and r["negation pass"] < 1.0:
            notes.append(f"**{r['model']}:** negation pair test pass rate is {r['negation pass']:.0%}. Review the failing wording.")
    if any(r["model"] == "bert" for r in rows) and any(r["model"] == "baseline" for r in rows):
        b = next(r for r in rows if r["model"] == "baseline"); m = next(r for r in rows if r["model"] == "bert")
        notes.append(f"**Baseline vs BERT:** false alerts safely down-ranked {b['false alerts low-priority']:.1%} vs {m['false alerts low-priority']:.1%} "
                     "at the recall gate. If the baseline matches BERT, ship the simpler model.")
    display(Markdown("### Reading the results\n" + "\n".join(f"- {n}" for n in notes)))

### Model comparison on the held-out test set

model,baseline,bert
gate met (val),True,True
threshold,0.1017,0.0343
test recall,0.9969,1.0
recall lower bound,0.9904,0.9954
missed distress,2,0
false alerts low-priority,0.892,0.889
negation pass,1.0,0.8
slices FAIL,6,0
slices INSUFFICIENT,7,11
meets release criteria,False,True


### Slice gate: baseline

,slice,value,n,n_pos,recall,recall_lb,missed,low_priority_frac_of_false,gate
0,language,en,2074,445,0.996,0.986,2,0.891,FAIL
1,language,es,471,115,1.000,0.974,0,0.893,INSUFFICIENT_DATA
2,language,fr,453,91,1.000,0.968,0,0.898,INSUFFICIENT_DATA
3,beacon_type,ELT,902,207,1.000,0.986,0,0.898,INSUFFICIENT_DATA
4,beacon_type,EPIRB,1206,262,1.000,0.989,0,0.890,INSUFFICIENT_DATA
5,beacon_type,PLB,890,182,0.989,0.966,2,0.890,FAIL
6,source_mix,asr,1176,240,1.000,0.988,0,0.931,INSUFFICIENT_DATA
7,source_mix,typed,1822,411,0.995,0.985,2,0.867,FAIL
8,administration,ADM_A,957,187,0.995,0.975,1,0.895,FAIL
9,administration,ADM_B,1048,234,1.000,0.987,0,0.883,INSUFFICIENT_DATA


### Slice gate: bert

,slice,value,n,n_pos,recall,recall_lb,missed,low_priority_frac_of_false,gate
0,language,en,2074,445,1.000,0.993,0,0.885,PASS
1,language,es,471,115,1.000,0.974,0,0.899,INSUFFICIENT_DATA
2,language,fr,453,91,1.000,0.968,0,0.898,INSUFFICIENT_DATA
3,beacon_type,ELT,902,207,1.000,0.986,0,0.898,INSUFFICIENT_DATA
4,beacon_type,EPIRB,1206,262,1.000,0.989,0,0.887,INSUFFICIENT_DATA
5,beacon_type,PLB,890,182,1.000,0.984,0,0.884,INSUFFICIENT_DATA
6,source_mix,asr,1176,240,1.000,0.988,0,0.928,INSUFFICIENT_DATA
7,source_mix,typed,1822,411,1.000,0.993,0,0.863,PASS
8,administration,ADM_A,957,187,1.000,0.984,0,0.890,INSUFFICIENT_DATA
9,administration,ADM_B,1048,234,1.000,0.987,0,0.886,INSUFFICIENT_DATA


### Reading the results
- **These results come from SYNTHETIC data.** They only confirm that the pipeline runs end to end. They say nothing about real-world performance or safety.
- **baseline:** 6 slice(s) FAIL the recall gate. Each is a release blocker.
- **baseline:** 7 slice(s) have too few positives to prove the gate. Collect more data or document the risk.
- **bert:** 11 slice(s) have too few positives to prove the gate. Collect more data or document the risk.
- **bert:** negation pair test pass rate is 80%. Review the failing wording.
- **Baseline vs BERT:** false alerts safely down-ranked 89.2% vs 88.9% at the recall gate. If the baseline matches BERT, ship the simpler model.

## 10. Smoke test: load the saved model the way the service does

In [10]:
import sys
sys.path.insert(0, "/content/alert_triage")
from triage.serve import load_predictor

if os.path.exists("artifacts/bert/operating_point.json"):
    pred, op = load_predictor("artifacts/bert/operating_point.json")
    samples = [
        "[META] EPIRB sea stationary registered=True [OWNER_CALL] Owner states beacon was accidentally activated, all persons safe.",
        "[META] EPIRB sea moving registered=True [OWNER_CALL] Owner could not be reached by phone.",
        "[META] ELT air moving registered=True [RCC_LOG] Emergency contact confirms aircraft is overdue.",
        "[META] PLB land moving registered=False [RADIO_RELAY] Persons are not safe, two people in the water.",
    ]
    probs = pred.predict_proba(samples)
    for text, p in zip(samples, probs):
        label = "PRIORITIZE" if p >= op["threshold"] else "LOW_PRIORITY_SUGGESTED"
        print(f"{label:24s} p={p:.3f}  {text[-80:]}")
    print("threshold =", round(op["threshold"], 4))
else:
    print("No BERT artifacts yet.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LOW_PRIORITY_SUGGESTED   p=0.005  e [OWNER_CALL] Owner states beacon was accidentally activated, all persons safe.
PRIORITIZE               p=0.720  IRB sea moving registered=True [OWNER_CALL] Owner could not be reached by phone.
PRIORITIZE               p=0.998  moving registered=True [RCC_LOG] Emergency contact confirms aircraft is overdue.
PRIORITIZE               p=0.998  ng registered=False [RADIO_RELAY] Persons are not safe, two people in the water.
threshold = 0.0343


## 11. Save results (Drive, or download a zip)

In [11]:
import zipfile

def collect_files(root="artifacts"):
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "ckpt"]       # skip large training checkpoints
        for f in filenames:
            yield os.path.join(dirpath, f)

if USE_DRIVE:
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    for src in collect_files():
        dst = os.path.join(DRIVE_OUT_DIR, src)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
    print("Saved to Drive:", DRIVE_OUT_DIR)
else:
    from google.colab import files
    with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in collect_files():
            z.write(f)
    print("results.zip size: %.0f MB" % (os.path.getsize("results.zip") / 1e6))
    files.download("results.zip")

Saved to Drive: /content/drive/MyDrive/alert_triage/results


## 12. Optional cleanup
If you used real data, delete it from the runtime when you are done. Use **Runtime → Disconnect and delete runtime** as well.

In [12]:
CLEAN_UP = False   #@param {type:"boolean"}
if CLEAN_UP:
    shutil.rmtree("data", ignore_errors=True)
    print("Deleted data/")
else:
    print("Skipped. Set CLEAN_UP = True to delete the data folder.")

Skipped. Set CLEAN_UP = True to delete the data folder.


## Using the results locally (Windows, CPU)
Unzip `results.zip` into the project root so you have `artifacts/bert/final` and `artifacts/bert/operating_point.json`, then:

```powershell
pip install -r requirements-train.txt
$env:OPERATING_POINT = "artifacts/bert/operating_point.json"
uvicorn triage.serve:app --port 8080
```